In [3]:
import pandas as pd
import numpy as np



In [4]:
df = pd.read_csv("powerplant.csv")

In [5]:
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [6]:
import torch
import torch.nn as nn

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

x = df.drop(["PE"],axis=1)
y = df["PE"]

x_train,x_test,y_train,y_test=train_test_split(
    x,y,test_size=0.2,random_state=42
)



In [8]:
scaler = StandardScaler()
x_train_scale = scaler.fit_transform(x_train)
x_test_scale = scaler.transform(x_test)

In [9]:
x_train_tensor = torch.tensor(x_train_scale,dtype=torch.float32)
x_test_tensor = torch.tensor(x_test_scale,dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)

y_test_tensor = torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)


In [10]:
from torch.utils.data import TensorDataset,DataLoader

train_dataset=TensorDataset(x_train_tensor,y_train_tensor)
test_dataset=TensorDataset(x_test_tensor,y_test_tensor)

In [11]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=32)
test_loader=DataLoader(test_dataset,batch_size=32)

In [12]:
###actual deeeeeeeep learning


In [13]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()
        self.model=nn.Sequential(
            nn.Linear(x_train.shape[1],6),
            nn.ReLU(),

            nn.Linear(6,6),
            nn.ReLU(),

            nn.Linear(6,1),
        )
    def forward(self,x):
        return self.model(x)

In [14]:
import torch.optim as optim

In [15]:
model = ANN()

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [16]:
##training mode

In [24]:
train_losses=[]
val_losses = []
best_loss = float("inf")
epochs =100
for epoch in range(epochs):
    running_loss=0
    model.train()
    for xb,yb in train_loader:
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs,yb)
        loss.backward()
        optimizer.step() #params update
        running_loss+=loss.item()

    epoch_training_loss = running_loss/len(train_loader)
    train_losses.append(epoch_training_loss)

    ##validatation

    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for xb,yb in test_loader:
            outputs = model(xb)
            loss = criterion(outputs,yb)
            running_loss+=loss.item()

    epochs_val_loss = running_loss / len(test_loader)
    val_losses.append(epochs_val_loss)

    if best_loss > epochs_val_loss:
        best_loss = epochs_val_loss
        torch.save(model.state_dict(),"best_model.pt")

        
    

In [25]:
##loading

In [26]:
model.load_state_dict(torch.load("best_model.pt"))

<All keys matched successfully>

In [27]:
##evaluate

In [29]:
with torch.no_grad():

    train_preds = model(x_train_tensor)
    test_preds = model(x_test_tensor)

    train_mse_loss = criterion(
        train_preds,
        y_train_tensor
    )

    test_mse_loss = criterion(
        test_preds,
        y_test_tensor
    )

print("Training MSE:", train_mse_loss.item())
print("Testing MSE:", test_mse_loss.item())


from sklearn.metrics import r2_score

print(
    "r^2 score =",
    r2_score(y_test, test_preds.numpy())
)

Training MSE: 20.390064239501953
Testing MSE: 18.676515579223633
r^2 score = 0.9347303943268555


In [30]:
predicted_df = pd.DataFrame(
    test_preds.numpy(),
    columns=["Predicted Values"]
)

actual_df = pd.DataFrame(
    y_test.values,
    columns=["Actual Values"]
)

pd.concat(
    [predicted_df, actual_df],
    axis=1
)

,Predicted Values,Actual Values
0,435.095215,433.27
1,436.687378,438.16
2,461.546539,458.42
3,476.489899,480.82
4,435.411957,441.41
...,...,...
1909,451.180786,456.70
1910,431.421417,438.04
1911,467.554718,467.80
1912,430.855743,437.14
